# lib_engine — 공통 엔진 (logit 담당 P2)
함수 정의만 둡니다. 다른 노트북은 아래처럼 불러 씁니다 (설계도 약속).
```python
SKIP_LIB_TESTS = True
%run -i ./lib_engine.ipynb
```
이 노트북을 직접 열어 실행하면 맨 아래 점검이 돕니다. 이름과 인자는 팀 약속입니다. 바꾸려면 먼저 팀에 알립니다.
같은 내용의 `lib_engine.py`도 있습니다 (`from lib_engine import *`). 두 파일은 `lib_engine.py`에서 만들어지므로 고칠 때는 `.py`를 고치고 다시 만드세요.

**v1 변경 (팀 공유)**
- `sr_curve` 반환에 `rate_eff` 열 추가 (p̂ ≤ τ 규칙의 실제 승인 비율. 동점이 많으면 rate보다 큼). `stack_curves`는 기본으로 계약 열(b, rate, sr, tau)만 냄
- `pick_cutoff`: 유효한 SR이 없으면 에러 대신 (nan, nan, nan)
- `aggregate_bootstrap`: a_star가 nan인 회차는 요약에서 빼고 `n_failed`로 셈
- `set_korean_font` 추가, fig_a/b/c가 자동 호출
- `RATES`는 00_config에 있으면 그 값을 씀
- `aggregate_bootstrap` 요약에 `more_stable` (a*_b와 τ*_b 중 변동계수가 작은 쪽)
- `fit_on(prep, X_raw, train_idx)`, `assert_fit_on(prep, train_idx)`: 전처리를 Train_b에서만 fit했는지 점검 (②③도 같이 쓰면 됨)
- `split_fingerprint(splits, n_samples)`: 분할 지문(12자). summary.json의 `split_hash`로 남겨 06_collect가 ①②③ 분할이 같은지 비교

TODO(질문): `make_oob_splits`가 model_template의 것과 같은 분할을 내는지 확인 (02_check_p2가 model_template.ipynb가 있으면 자동 비교).

In [ ]:
import numpy as np
import pandas as pd


## 1. 표본 분할

In [ ]:
def make_oob_splits(n_samples, n_bootstrap, seed=42):
    """B회 복원추출. 반환: [(train_idx, oob_idx), ...]
    train_idx는 중복 포함 길이 n, oob_idx는 한 번도 안 뽑힌 행.
    호출: make_oob_splits(n_samples=len(Dev), n_bootstrap=B, seed=42). 세 모형이 같은 splits를 쓴다.
    model_template의 make_oob_splits와 같은 분할 (같은 n, seed면 같은 결과)."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_bootstrap):
        tr = rng.integers(0, n_samples, size=n_samples, dtype=np.int64)   # model_template과 같은 호출
        mask = np.ones(n_samples, dtype=bool)
        mask[tr] = False
        out.append((tr, np.flatnonzero(mask)))
    return out

def split_fingerprint(splits, n_samples):
    """분할 지문: 회차별 train_idx(중복 포함, 뽑힌 순서 그대로)와 n_samples로 만든 sha1 앞 12자.
    oob_idx는 train_idx에서 정해지므로 train_idx만 넣는다. ①②③이 같은 n, B, seed로 분할했으면 같은 값이 나온다.
    summary.json의 split_hash로 남기고, 06_collect가 세 모형의 값이 같은지 비교한다."""
    import hashlib
    h = hashlib.sha1()
    h.update(np.int64(n_samples).tobytes())
    h.update(np.int64(len(splits)).tobytes())
    for tr, _ in splits:
        h.update(np.ascontiguousarray(tr, dtype=np.int64).tobytes())
    return h.hexdigest()[:12]


## 1-1. 전처리 fit 위치 점검 (공통 명세 2: Train_b에서만 fit)
①②③ 모두 같은 방식으로 쓰도록 엔진에 둡니다. fit한 행의 지문을 남겨 Train_b가 아닌 데이터로 fit하면 assert가 실패합니다.

In [ ]:
def _idx_fingerprint(idx):
    import hashlib
    idx = np.ascontiguousarray(idx, dtype=np.int64)
    return (len(idx), hashlib.sha1(idx.tobytes()).hexdigest())


def fit_on(prep, X_raw, train_idx):
    """fit 전 전처리기 prep을 X_raw.iloc[train_idx](중복 포함 그대로)에만 fit하고 fit 위치 지문을 남긴다."""
    prep.fit(X_raw.iloc[train_idx])
    prep._fit_fp = _idx_fingerprint(train_idx)
    return prep


def assert_fit_on(prep, train_idx):
    """prep이 정확히 train_idx로 fit됐는지 확인. 아니면 AssertionError."""
    assert getattr(prep, "_fit_fp", None) == _idx_fingerprint(train_idx), "전처리가 Train_b가 아닌 데이터로 fit됨"


## 2. Sharpe 비율과 승인율별 SR 곡선
- 대출 단위 횡단면, 동일가중, ddof=1, 다시 연율화하지 않음 (excess_ret은 이미 연율화됨)
- `approved_only`: 승인된 대출만. `with_treasury`: 거절분의 초과수익률을 0(국채)으로 두고 전체 n건

In [ ]:
def sharpe(r):
    r = np.asarray(r, dtype=float)
    if len(r) < 2:
        return np.nan
    sd = r.std(ddof=1)
    return np.nan if sd == 0 else r.mean() / sd


def n_approved(rate, n):
    return max(int(round(rate * n)), 2)


def sr_curve(p, excess_ret, rates=None, mode=None):
    """p̂가 낮은 순서로 승인율 a만큼 승인했을 때의 SR.
    rates, mode를 생략하면 00_config의 RATES, SR_MODE를 쓴다 (model_template과 같은 기본값).
    반환: DataFrame[rate, sr, tau, rate_eff]
      tau      = 승인된 마지막 대출의 p̂
      rate_eff = 'p̂ <= tau면 승인' 규칙을 적용했을 때의 실제 승인 비율.
                 p̂ 동점(트리 모형 등)이 많으면 rate보다 커지므로 운영 컷오프 해석 시 확인할 것."""
    rates = RATES if rates is None else rates
    mode = globals().get("SR_MODE") if mode is None else mode
    assert mode in ("approved_only", "with_treasury"), "mode가 없거나 잘못됨: %r (00_config의 SR_MODE 또는 직접 지정)" % (mode,)
    p = np.asarray(p, dtype=float)
    r = np.asarray(excess_ret, dtype=float)
    n = len(p)
    order = np.argsort(p, kind="stable")
    ps, rs = p[order], r[order]
    rows = []
    for a in rates:
        k = min(n_approved(a, n), n)
        if mode == "approved_only":
            sr = sharpe(rs[:k])
        else:
            full = np.zeros(n)
            full[:k] = rs[:k]
            sr = sharpe(full)
        tau = ps[k - 1]
        rate_eff = np.searchsorted(ps, tau, side="right") / n
        rows.append((a, sr, tau, rate_eff))
    return pd.DataFrame(rows, columns=["rate", "sr", "tau", "rate_eff"])


def pick_cutoff(curve, min_rate=None):
    """min_rate 이상에서 SR 최대인 승인율. 반환: (a, tau, sr). 유효한 SR이 없으면 (nan, nan, nan)
    min_rate를 생략하면 00_config의 MIN_RATE를 쓴다."""
    min_rate = globals().get("MIN_RATE") if min_rate is None else min_rate
    assert min_rate is not None, "min_rate가 없음 (00_config의 MIN_RATE 또는 직접 지정)"
    c = curve[curve["rate"] >= min_rate - 1e-12].dropna(subset=["sr"])
    if c.empty:
        return np.nan, np.nan, np.nan
    row = c.loc[c["sr"].idxmax()]
    return float(row["rate"]), float(row["tau"]), float(row["sr"])


## 3. B회 집계
회차별 결과 `results`: 리스트의 각 원소는 dict(b, a_star, tau_star, sr_max, auc, brier, ...)

In [ ]:
# 공통 명세 4: RATES = 0.05, 0.06, ..., 0.95. 00_config에 RATES가 있으면 그 값을 쓴다 (%run -i는 같은 이름공간).
RATES = np.asarray(globals().get("RATES", np.round(np.arange(0.05, 0.9501, 0.01), 2)), dtype=float)


def aggregate_bootstrap(results):
    """bootstrap_results.csv용 표와 요약. 반환: (df, summary dict)
    df는 모든 회차를 그대로 담고, 요약은 a_star가 유효한 회차만으로 계산한다.
    summary: a_star(중앙값), a/tau 변동계수(어느 쪽이 회차 간 더 안정적인지 비교)"""
    df = pd.DataFrame(results)
    ok = df.dropna(subset=["a_star"])
    cv = lambda s: float(s.std(ddof=1) / abs(s.mean())) if len(s) > 1 and s.mean() != 0 else np.nan
    summary = {
        "a_star": float(ok["a_star"].median()) if len(ok) else np.nan,
        "a_star_cv": cv(ok["a_star"]),
        "tau_star_cv": cv(ok["tau_star"]),
        "sr_max_mean": float(ok["sr_max"].mean()) if len(ok) else np.nan,
        "B": int(len(df)),
        "n_failed": int(len(df) - len(ok)),
    }
    ca, ct = summary["a_star_cv"], summary["tau_star_cv"]
    summary["more_stable"] = (np.nan if not (np.isfinite(ca) and np.isfinite(ct))
                              else ("a" if ca < ct else "tau"))   # 회차 간 더 안정적인 커트라인 단위
    return df, summary


CURVE_COLS = ["b", "rate", "sr", "tau"]   # bootstrap_curves.csv 계약 열


def stack_curves(curves, extra=False):
    """회차별 곡선 리스트 -> bootstrap_curves.csv 형식 (b, rate, sr, tau).
    extra=True면 계약 밖 진단 열(rate_eff)도 붙인다. 진단은 별도 파일로 저장할 것."""
    cols = CURVE_COLS + ([c for c in curves[0].columns if c not in CURVE_COLS] if extra else [])
    return pd.concat([c.assign(b=i) for i, c in enumerate(curves)], ignore_index=True)[cols]


## 4. Figure A·B·C

In [ ]:
def set_korean_font():
    """그림의 한글이 네모로 깨지지 않게 설치된 한글 글꼴을 찾아 지정한다.
    Colab에는 한글 글꼴이 없으므로 먼저 `!apt-get -qq install fonts-nanum`을 실행한다
    (아래에서 파일을 직접 등록하므로 런타임 재시작은 필요 없음)."""
    import os
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    for path in ("/usr/share/fonts/truetype/nanum/NanumGothic.ttf",):
        if os.path.exists(path):
            font_manager.fontManager.addfont(path)
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in ("AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
        if name in have:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


def fig_a(curves_df, a_star=None, ax=None, label=None):
    """Figure A: 승인율별 SR 평균과 ±1SD (B=1이면 띠 없음)"""
    import matplotlib.pyplot as plt
    set_korean_font()
    ax = ax or plt.gca()
    g = curves_df.groupby("rate")["sr"]
    m, s = g.mean(), g.std(ddof=1)
    ax.plot(m.index, m.values, label=label)
    ax.fill_between(m.index, m - s, m + s, alpha=0.2)
    if a_star is not None and np.isfinite(a_star):
        ax.axvline(a_star, ls="--", c="gray")
    ax.set_xlabel("승인율 a"); ax.set_ylabel("SR")
    return ax


def fig_b(results_df, ax=None):
    """Figure B: 회차별 a*_b 분포와 SR*_b 분포"""
    import matplotlib.pyplot as plt
    set_korean_font()
    if ax is None:
        _, ax = plt.subplots(1, 2, figsize=(9, 3.2))
    ax[0].hist(results_df["a_star"].dropna(), bins=20); ax[0].set_xlabel("a*_b")
    ax[1].hist(results_df["sr_max"].dropna(), bins=20); ax[1].set_xlabel("SR*_b")
    return ax


def fig_c(curves_ref, curves_new, ax=None):
    """Figure C: 회차별 ΔSR_b (ref − new) 곡선. 두 curves_df는 같은 b와 rate를 가져야 함"""
    import matplotlib.pyplot as plt
    set_korean_font()
    ax = ax or plt.gca()
    m = curves_ref[["b", "rate", "sr"]].merge(curves_new[["b", "rate", "sr"]], on=["b", "rate"], suffixes=("_r", "_n"))
    m["d"] = m["sr_r"] - m["sr_n"]
    for _, g in m.groupby("b"):
        ax.plot(g["rate"], g["d"], c="gray", alpha=0.2, lw=0.8)
    ax.plot(m.groupby("rate")["d"].mean(), c="C0")
    ax.axhline(0, c="k", lw=0.8)
    ax.set_xlabel("승인율 a"); ax.set_ylabel("ΔSR")
    return ax


## 5. 단위 테스트 (lib를 직접 열 때만 실행)
함수가 정의대로 계산하는지만 보는 합성 배열 테스트입니다. 모형 학습용 가짜 데이터가 아닙니다.
실제 Dev 완료 테스트(곡선 모양, 승인율 1.00, 전처리 fit 위치)는 02_logit.ipynb 9절에 있습니다.

In [ ]:
def _self_test():
    rng = np.random.default_rng(42)
    n = 20000
    p_true = rng.beta(1.2, 5.5, n)          # 연속형이라 동점 없음
    y = rng.random(n) < p_true
    excess = np.where(y, -0.55 + 0.15 * rng.standard_normal(n), 0.10 + 0.05 * rng.standard_normal(n))

    # (1) 행 순서를 섞어도 같은 곡선 (동점 없는 p̂ 기준)
    c1 = sr_curve(p_true, excess, RATES, "approved_only")
    perm = rng.permutation(n)
    c2 = sr_curve(p_true[perm], excess[perm], RATES, "approved_only")
    pd.testing.assert_frame_equal(c1, c2)

    # (2) 정답 p̂로 낮은 순 승인 -> 좋은 대출부터 들어오므로 승인율이 낮을 때 SR이 높을 때보다 큼
    #     a*의 위치(안쪽인지 하한인지)는 데이터와 SR 정의에 따라 달라서 점검하지 않고 출력만 한다
    #     (설계도 결정 기록: 승인만 기준 a*는 하한에 걸릴 수 있음)
    a, tau, sr = pick_cutoff(c1, 0.05)
    print("approved_only  a*=%.2f tau=%.3f SR=%.3f" % (a, tau, sr))
    assert c1["sr"].iloc[0] > c1["sr"].iloc[-1], (c1["sr"].iloc[0], c1["sr"].iloc[-1])
    ct = sr_curve(p_true, excess, RATES, "with_treasury")
    a2, _, sr2 = pick_cutoff(ct, 0.05)
    print("with_treasury  a*=%.2f SR=%.3f" % (a2, sr2))
    assert c1["tau"].is_monotonic_increasing

    # (2a) 승인율 1.00에서는 거절이 없으므로 두 모드의 SR이 같다 (완료 테스트 항목)
    s_ao = sr_curve(p_true, excess, [1.0], "approved_only")["sr"].iloc[0]
    s_wt = sr_curve(p_true, excess, [1.0], "with_treasury")["sr"].iloc[0]
    assert np.isclose(s_ao, s_wt), (s_ao, s_wt)
    assert np.allclose(c1["rate_eff"], [min(n_approved(r, n), n) / n for r in RATES])  # 동점 없으면 rate와 같음

    # (2b) 동점이 많으면 rate_eff > rate
    cq = sr_curve(np.round(p_true, 1), excess, RATES, "approved_only")
    assert (cq["rate_eff"] >= cq["rate"] - 1e-9).all() and (cq["rate_eff"] > cq["rate"] + 0.01).any()

    # (3) 무작위 p̂면 SR이 낮음 (정보 없음)
    cr = sr_curve(rng.random(n), excess, RATES, "approved_only")
    assert cr["sr"].max() < c1["sr"].max()

    # (4) 분할: OOB는 train에 없음, 비율 약 36.8%
    sp = make_oob_splits(n, 3, seed=42)
    assert all(len(set(tr) & set(oob)) == 0 for tr, oob in sp)
    print("OOB 비율 %.3f" % np.mean([len(o) / n for _, o in sp]))
    assert make_oob_splits(n, 3, 42)[0][0].tolist() == sp[0][0].tolist()  # 재현성
    # (4a) 분할 지문: 같은 n·B·seed면 같고, seed·B·n이 다르면 다름
    fp = split_fingerprint(sp, n)
    assert fp == split_fingerprint(make_oob_splits(n, 3, 42), n) and len(fp) == 12
    assert fp != split_fingerprint(make_oob_splits(n, 3, 43), n)
    assert fp != split_fingerprint(make_oob_splits(n, 2, 42), n)
    assert fp != split_fingerprint(make_oob_splits(n - 1, 3, 42), n - 1)

    # (5) SR이 전부 nan이면 pick_cutoff가 멈추지 않고 nan을 돌려줌
    assert all(np.isnan(pick_cutoff(c1.assign(sr=np.nan), 0.05)))

    # (6) 집계: 실패 회차는 요약에서 빠지고 n_failed로 셈
    res = [dict(b=i, a_star=a + 0.01 * i, tau_star=tau, sr_max=sr, auc=0.7, brier=0.1) for i in range(5)]
    res.append(dict(b=5, a_star=np.nan, tau_star=np.nan, sr_max=np.nan, auc=0.7, brier=0.1))
    df, summ = aggregate_bootstrap(res); print(summ)
    assert summ["n_failed"] == 1 and summ["B"] == 6 and np.isclose(summ["a_star"], a + 0.02)
    assert summ["more_stable"] in ("a", "tau")

    # (7) 전처리 fit 위치: Train_b로 fit하면 통과, 다른 데이터로 fit하면 assert 실패
    from sklearn.preprocessing import StandardScaler
    Xd = pd.DataFrame({"a": rng.normal(size=200), "b": rng.normal(size=200)})
    tr, _ = make_oob_splits(n_samples=200, n_bootstrap=1, seed=1)[0]
    assert_fit_on(fit_on(StandardScaler(), Xd, tr), tr)
    try:
        assert_fit_on(fit_on(StandardScaler(), Xd, np.arange(200)), tr)
        raise SystemExit("assert가 안 터짐")
    except AssertionError:
        pass
    print("OK")


In [ ]:
if not globals().get("SKIP_LIB_TESTS", False):
    _self_test()
